# NB07 — Sensitivity and Robustness Checks

This notebook audits the NB02 main findings (metal → CWM at L1) for three key vulnerabilities:

1. **pESS (pseudo effective sample size)** — spatial autocorrelation inflates effective degrees of freedom. We compute Moran's I on metal residuals and report n_eff = n(1−I)/(1+I).
2. **Permutation test** — empirical null distribution: shuffle metal vector 500×, rerun FWL, compare observed FDR hit count to null.
3. **Collider sensitivity** — KOs that appear at L5 but not L1 may reflect collider induction by community composition covariates. We flag these and inspect direction changes.
4. **pH sensitivity** — linear vs natural spline pH: do the 382 L1 hits survive both forms?
5. **Metal transformation** — log₁₀ vs rank-normalization: does the hit set change substantially?

All tests run on the USA 634-cell dataset (same as NB02).


In [1]:
import sys
sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, METAL_COLORS, FIGW, ROW_H
apply_style()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t as t_dist, spearmanr, rankdata
import patsy
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/data')
FIGS = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/figures')
FIGS.mkdir(exist_ok=True)

METALS = ['As','Cd','Cr','Cu','Ni','Pb','Zn']


In [2]:
# Load NB02 inputs: CWM wide matrix, metal concentrations, thinned sample metadata
nb00     = pd.read_parquet(DATA / 'nb00_thinned_samples.parquet')
combined = pd.read_parquet(DATA / 'nb02_combined_metals.parquet')
soil     = pd.read_parquet(DATA / 'nb02_soil_props.parquet')
lith     = pd.read_parquet(DATA / 'nb02_lithology.parquet')
wclim    = pd.read_parquet(DATA / 'nb02_worldclim_seasonal.parquet')
gc       = pd.read_parquet(DATA / 'nb01_genus_counts.parquet')
gp       = pd.read_parquet(DATA / 'nb02_genus_phylum.parquet')
fdr_all  = pd.read_parquet(DATA / 'nb02_fwl_results_fdr.parquet')
nb02_L1  = fdr_all[(fdr_all['level']=='L1') & (fdr_all['q_bh']<0.05)]
nb02_L5  = fdr_all[(fdr_all['level']=='L5') & (fdr_all['q_bh']<0.05)]

# CWM — all thinned samples
cwm_long = pd.read_parquet(DATA / 'nb01_cwm.parquet')
cwm_wide = cwm_long.pivot_table(index='sample_id', columns='ko_id', values='cwm', fill_value=0.0)

# Metal concentrations index — all regions (same as NB02)
combined_idx = combined.set_index('sample_id')

# Build base DataFrame — ALL thinned samples (NB02 uses all 4884, filters by metal availability in FWL)
tot = gc.groupby('sample_id')['genus_count'].sum().rename('total')
gc2 = gc.join(tot, on='sample_id'); gc2['ra'] = gc2['genus_count']/gc2['total']
gc2['h'] = -gc2['ra']*np.log(gc2['ra'].clip(lower=1e-12))
shannon = gc2.groupby('sample_id')['h'].sum().rename('shannon')
gc2 = gc2.merge(gp, on='genus_lower', how='left')
gc2['phylum_lower'] = gc2['phylum_lower'].fillna('unknown')
phyl_ra = gc2.groupby(['sample_id','phylum_lower'])['ra'].sum().reset_index()
top_phyla = phyl_ra.groupby('phylum_lower')['ra'].mean().nlargest(8).index.tolist()
phyl_wide = (phyl_ra[phyl_ra['phylum_lower'].isin(top_phyla)]
             .pivot_table(index='sample_id', columns='phylum_lower', values='ra', fill_value=0.0))
phyl_wide.columns = [f'phyl_{c}' for c in phyl_wide.columns]
phyl_cols = phyl_wide.columns.tolist()

base = (nb00
        .merge(soil, on='sample_id', how='left')
        .merge(lith, on='sample_id', how='left')
        .merge(wclim, on='sample_id', how='left')
        .merge(shannon.reset_index(), on='sample_id', how='left')
        .merge(phyl_wide.reset_index(), on='sample_id', how='left')
        .set_index('sample_id'))
base[phyl_cols] = base[phyl_cols].fillna(0.0)
cwm_wide = cwm_wide.reindex(base.index).fillna(0.0)
ko_ids = cwm_wide.columns.tolist()

print(f'base: {base.shape}, CWM: {cwm_wide.shape}')
print(f'NB02 L1 hits: {len(nb02_L1)}, L5 hits: {len(nb02_L5)}')

# Compute ph_best: measured > OLM÷10 > SoilGrids raster (covers 966/1006 otherwise-missing)
base['ph_raw'] = pd.to_numeric(base['ph'], errors='coerce')
base['ph_olm'] = pd.to_numeric(base['olm_soil_ph_0cm_H2O'], errors='coerce') / 10.0
base['ph_best'] = base['ph_raw'].where(base['ph_raw'].notna(),
                  base['ph_olm'].where(base['ph_olm'].notna(), base['ph_soilgrids']))
print(f"ph_best: {base['ph_best'].notna().sum()} non-null, range {base['ph_best'].min():.1f}–{base['ph_best'].max():.1f}")


base: (4884, 33), CWM: (4884, 6557)
NB02 L1 hits: 560, L5 hits: 51
ph_best: 4844 non-null, range 2.5–10.4


In [3]:
# FWL engine — copied from NB02 to ensure identical results
def fwl_all_kos(X, Y, Z):
    n, p = Z.shape
    coef_x, *_ = np.linalg.lstsq(Z, X, rcond=None)
    Mx = X - Z @ coef_x
    coef_y, *_ = np.linalg.lstsq(Z, Y, rcond=None)
    My = Y - Z @ coef_y
    MxMx = Mx @ Mx
    betas = My.T @ Mx / MxMx
    resid = My - np.outer(Mx, betas)
    dof = max(n - p - 1, 1)
    sigma2 = (resid**2).sum(axis=0) / dof
    se = np.sqrt(sigma2 / MxMx)
    t_stat = betas / np.where(se > 0, se, np.nan)
    return betas, se, t_stat

def build_Z_spline(df, level):
    """L-level Z with pH as natural spline (matches NB02)."""
    n = len(df); Z = np.ones((n, 1))
    if level == 0: return Z
    ph = pd.to_numeric(df['ph_best'], errors='coerce').fillna(df['ph_best'].median() if df['ph_best'].notna().any() else 6.5).values
    dm = patsy.dmatrix('cr(ph, df=3) - 1', {'ph': ph}, return_type='matrix')
    Z = np.hstack([Z, np.array(dm)])
    if level == 1: return Z
    for c in ['clay_pct','som_pct','bulk_density']:
        col = pd.to_numeric(df[c], errors='coerce').values
        col = np.where(np.isfinite(col), col, np.nanmedian(col) if np.isfinite(col).any() else 0.0)
        Z = np.hstack([Z, col.reshape(-1,1)])
    lith_dummies = pd.get_dummies(df.get('lith_class','unknown'), prefix='lith').values.astype(float)
    Z = np.hstack([Z, lith_dummies])
    if level == 2: return Z
    lights = pd.to_numeric(df.get('lights_radiance_nanow_cm2_sr', pd.Series(dtype=float)), errors='coerce').values
    lights = np.where(np.isfinite(lights), np.log10(lights+0.1), 0.0)
    Z = np.hstack([Z, lights.reshape(-1,1)])
    if level == 3: return Z
    for c in ['era5_total_precipitation_mm','temp_seasonality','precip_seasonality','temp_range']:
        col = pd.to_numeric(df.get(c, pd.Series(dtype=float)), errors='coerce').values
        col = np.where(np.isfinite(col), col, 0.0)
        Z = np.hstack([Z, col.reshape(-1,1)])
    if level == 4: return Z
    sh = pd.to_numeric(df['shannon'], errors='coerce').values
    sh = np.where(np.isfinite(sh), sh, np.nanmedian(sh) if np.isfinite(sh).any() else 0.0)
    Z = np.hstack([Z, sh.reshape(-1,1)])
    for c in phyl_cols:
        if c in df.columns:
            col = df[c].values.astype(float)
            Z = np.hstack([Z, col.reshape(-1,1)])
    return Z

def build_Z_linear(df, level):
    """L-level Z with linear pH (sensitivity check)."""
    n = len(df); Z = np.ones((n, 1))
    if level == 0: return Z
    ph = pd.to_numeric(df['ph_best'], errors='coerce').values
    ph = np.where(np.isfinite(ph), ph, np.nanmedian(ph) if np.isfinite(ph).any() else 6.5)
    Z = np.hstack([Z, ph.reshape(-1,1)])
    if level == 1: return Z
    for c in ['clay_pct','som_pct','bulk_density']:
        col = pd.to_numeric(df[c], errors='coerce').values
        col = np.where(np.isfinite(col), col, np.nanmedian(col) if np.isfinite(col).any() else 0.0)
        Z = np.hstack([Z, col.reshape(-1,1)])
    return Z   # truncate at L2 for this test

def run_fwl_metal(df, metal, level, cwm_mat, ko_list, build_Z_fn=None, X_override=None):
    """Run FWL for one metal, return results DataFrame."""
    if build_Z_fn is None: build_Z_fn = build_Z_spline
    if X_override is None:
        raw = pd.to_numeric(combined_idx.reindex(df.index)[metal], errors='coerce').values
        X = np.log10(np.where(raw > 0, raw, np.nan))
    else:
        X = X_override
    Z = build_Z_fn(df, level)
    valid = np.isfinite(X) & np.all(np.isfinite(Z), axis=1)
    n_valid = int(valid.sum())
    if n_valid < 30: return None
    betas, se, t_stat = fwl_all_kos(X[valid], cwm_mat[valid], Z[valid])
    dof = max(n_valid - Z.shape[1] - 1, 1)
    pvals = 2 * t_dist.sf(np.abs(t_stat), df=dof)
    fp = np.isfinite(pvals)
    q_full = np.full(len(ko_list), np.nan)
    if fp.sum() > 0:
        _, q_vals, _, _ = multipletests(pvals[fp], method='fdr_bh')
        q_full[fp] = q_vals
    return pd.DataFrame({'ko_id': ko_list, 'beta': betas, 'se': se, 't': t_stat,
                         'p': pvals, 'q_bh': q_full, 'n': n_valid, 'metal': metal})


In [4]:
# === 1. pESS: Moran's I on metal residuals (L1 FWL) ===
# n_eff = n * (1 - I) / (1 + I) where I = Moran's I on the L1-partialled metal vector.
# We use k=10 nearest-neighbor weights (computationally tractable for n~600).
from scipy.spatial import KDTree

coords = base[['lat','lon']].values
K = 10  # nearest neighbors for spatial weight matrix

def moran_knn(residuals, coords, k=10):
    """Moran's I via KNN spatial weights. Returns (I, n_eff)."""
    n = len(residuals)
    tree = KDTree(coords)
    _, idx = tree.query(coords, k=k+1)   # +1 because first neighbor is self
    idx = idx[:, 1:]   # exclude self
    x = residuals - residuals.mean()
    W_sum = n * k     # each point has k neighbors (row-standardized)
    moran_num = sum(x[i] * sum(x[idx[i]]) for i in range(n))
    moran_den = (x**2).sum()
    I = (n / W_sum) * (moran_num / moran_den) if moran_den > 0 else 0.0
    n_eff = n * (1 - I) / (1 + I)
    return I, max(n_eff, 1.0)

cwm_vals = cwm_wide.values
pess_rows = []
for metal in METALS:
    if metal not in combined_idx.columns: continue
    raw = pd.to_numeric(combined_idx.reindex(base.index)[metal], errors='coerce').values
    X = np.log10(np.where(raw > 0, raw, np.nan))
    Z1 = build_Z_spline(base, 1)
    valid = np.isfinite(X) & np.all(np.isfinite(Z1), axis=1)
    n_valid = int(valid.sum())
    if n_valid < 30: continue
    # Partial out Z from X (get metal residual after pH control)
    coef_x, *_ = np.linalg.lstsq(Z1[valid], X[valid], rcond=None)
    Mx = X[valid] - Z1[valid] @ coef_x
    coords_v = coords[valid]
    I, n_eff = moran_knn(Mx, coords_v, k=10)
    n_obs = n_valid
    print(f'{metal}: n={n_obs}, Moran I={I:.3f}, n_eff={n_eff:.0f}')
    pess_rows.append({'metal':metal,'n':n_obs,'moran_I':I,'n_eff':round(n_eff)})

pess_df = pd.DataFrame(pess_rows)
print('\n=== pESS summary ===')
print(pess_df.to_string(index=False))


As: n=1143, Moran I=0.448, n_eff=435
Cd: n=998, Moran I=0.397, n_eff=430
Cr: n=1693, Moran I=0.481, n_eff=593
Cu: n=1689, Moran I=0.377, n_eff=764
Ni: n=1645, Moran I=0.390, n_eff=722
Pb: n=1630, Moran I=0.354, n_eff=778
Zn: n=1184, Moran I=0.395, n_eff=513

=== pESS summary ===
metal    n  moran_I  n_eff
   As 1143 0.448472    435
   Cd  998 0.397346    430
   Cr 1693 0.480921    593
   Cu 1689 0.376973    764
   Ni 1645 0.390237    722
   Pb 1630 0.353793    778
   Zn 1184 0.395385    513


In [5]:
# === 2. Permutation test: shuffle metal → empirical null FDR hit count ===
# For each metal, permute the metal vector 500x, run FWL at L1, record FDR hit count.
# Compare observed hit count to null distribution.
N_PERM = 500
rng = np.random.default_rng(42)

cwm_vals = cwm_wide.values
perm_rows = []

for metal in METALS:
    if metal not in combined_idx.columns: continue
    raw = pd.to_numeric(combined_idx.reindex(base.index)[metal], errors='coerce').values
    X_log = np.log10(np.where(raw > 0, raw, np.nan))
    Z1 = build_Z_spline(base, 1)
    valid = np.isfinite(X_log) & np.all(np.isfinite(Z1), axis=1)
    n_valid = int(valid.sum())
    if n_valid < 30: continue
    X_v = X_log[valid]; Z_v = Z1[valid]; Y_v = cwm_vals[valid]
    
    # Observed hits
    betas_obs, se_obs, t_obs = fwl_all_kos(X_v, Y_v, Z_v)
    dof = max(n_valid - Z_v.shape[1] - 1, 1)
    pvals_obs = 2 * t_dist.sf(np.abs(t_obs), df=dof)
    fp = np.isfinite(pvals_obs)
    q_obs = np.full(len(ko_ids), np.nan)
    if fp.sum() > 0:
        _, q_vals, _, _ = multipletests(pvals_obs[fp], method='fdr_bh')
        q_obs[fp] = q_vals
    n_obs_hits = int((q_obs < 0.05).sum())
    
    # Permutation null
    null_hits = []
    for _ in range(N_PERM):
        X_perm = rng.permutation(X_v)
        betas_p, se_p, t_p = fwl_all_kos(X_perm, Y_v, Z_v)
        pvals_p = 2 * t_dist.sf(np.abs(t_p), df=dof)
        fp_p = np.isfinite(pvals_p)
        q_p = np.full(len(ko_ids), np.nan)
        if fp_p.sum() > 0:
            _, q_perm, _, _ = multipletests(pvals_p[fp_p], method='fdr_bh')
            q_p[fp_p] = q_perm
        null_hits.append(int((q_p < 0.05).sum()))
    
    null_arr = np.array(null_hits)
    perm_p = (null_arr >= n_obs_hits).mean()
    print(f'{metal}: observed={n_obs_hits}, null mean={null_arr.mean():.1f}±{null_arr.std():.1f}, perm p={perm_p:.4f}')
    perm_rows.append({'metal':metal,'n_obs':n_obs_hits,
                      'null_mean':null_arr.mean(),'null_sd':null_arr.std(),
                      'null_max':null_arr.max(),'perm_p':perm_p})

perm_df = pd.DataFrame(perm_rows)
print('\n=== Permutation test summary ===')
print(perm_df.to_string(index=False))
perm_df.to_parquet(DATA / 'nb07_perm_results.parquet', index=False)


As: observed=79, null mean=9.0±47.1, perm p=0.0200


Cd: observed=30, null mean=16.5±106.6, perm p=0.0640


Cr: observed=66, null mean=4.9±23.7, perm p=0.0220


Cu: observed=8, null mean=3.7±48.5, perm p=0.0220


Ni: observed=41, null mean=6.2±22.8, perm p=0.0440


Pb: observed=20, null mean=8.7±28.9, perm p=0.0880


Zn: observed=76, null mean=9.7±54.3, perm p=0.0300

=== Permutation test summary ===
metal  n_obs  null_mean    null_sd  null_max  perm_p
   As     79      8.968  47.116780       733   0.020
   Cd     30     16.492 106.555197      1337   0.064
   Cr     66      4.932  23.729631       361   0.022
   Cu      8      3.730  48.534247      1015   0.022
   Ni     41      6.176  22.796250       312   0.044
   Pb     20      8.698  28.913229       399   0.088
   Zn     76      9.688  54.266036       973   0.030


In [6]:
# === 3. pH sensitivity: linear pH vs spline pH ===
# Compare L1 FDR hits using linear vs spline pH control.
# If hit sets are near-identical, results are not sensitive to pH form.
cwm_vals = cwm_wide.values
ph_sens_rows = []

for metal in METALS:
    if metal not in combined_idx.columns: continue
    raw = pd.to_numeric(combined_idx.reindex(base.index)[metal], errors='coerce').values
    X_log = np.log10(np.where(raw > 0, raw, np.nan))
    
    # Spline (standard)
    Z_spl = build_Z_spline(base, 1)
    valid_spl = np.isfinite(X_log) & np.all(np.isfinite(Z_spl), axis=1)
    betas_s, se_s, t_s = fwl_all_kos(X_log[valid_spl], cwm_vals[valid_spl], Z_spl[valid_spl])
    dof = max(valid_spl.sum() - Z_spl.shape[1] - 1, 1)
    p_s = 2 * t_dist.sf(np.abs(t_s), df=dof)
    fp = np.isfinite(p_s)
    q_s = np.full(len(ko_ids), np.nan)
    if fp.sum() > 0:
        _, q_v, _, _ = multipletests(p_s[fp], method='fdr_bh'); q_s[fp] = q_v
    hits_spline = set(np.array(ko_ids)[q_s < 0.05])
    
    # Linear
    Z_lin = build_Z_linear(base, 1)
    valid_lin = np.isfinite(X_log) & np.all(np.isfinite(Z_lin), axis=1)
    betas_l, se_l, t_l = fwl_all_kos(X_log[valid_lin], cwm_vals[valid_lin], Z_lin[valid_lin])
    dof2 = max(valid_lin.sum() - Z_lin.shape[1] - 1, 1)
    p_l = 2 * t_dist.sf(np.abs(t_l), df=dof2)
    fp2 = np.isfinite(p_l)
    q_l = np.full(len(ko_ids), np.nan)
    if fp2.sum() > 0:
        _, q_v2, _, _ = multipletests(p_l[fp2], method='fdr_bh'); q_l[fp2] = q_v2
    hits_linear = set(np.array(ko_ids)[q_l < 0.05])
    
    ov = len(hits_spline & hits_linear)
    total = max(len(hits_spline), 1)
    pct = 100 * ov / total
    new_in_lin = len(hits_linear - hits_spline)
    new_in_spl = len(hits_spline - hits_linear)
    print(f'{metal}: spline={len(hits_spline)}, linear={len(hits_linear)}, '
          f'overlap={ov} ({pct:.0f}%), only-spline={new_in_spl}, only-linear={new_in_lin}')
    ph_sens_rows.append({'metal':metal,'n_spline':len(hits_spline),'n_linear':len(hits_linear),
                         'overlap':ov,'pct_overlap':pct,'only_spline':new_in_spl,'only_linear':new_in_lin})

ph_df = pd.DataFrame(ph_sens_rows)
print('\n=== pH form sensitivity summary ===')
print(ph_df[['metal','n_spline','n_linear','overlap','pct_overlap']].to_string(index=False))


As: spline=79, linear=95, overlap=79 (100%), only-spline=0, only-linear=16


Cd: spline=30, linear=118, overlap=30 (100%), only-spline=0, only-linear=88


Cr: spline=66, linear=68, overlap=65 (98%), only-spline=1, only-linear=3


Cu: spline=8, linear=8, overlap=8 (100%), only-spline=0, only-linear=0


Ni: spline=41, linear=43, overlap=41 (100%), only-spline=0, only-linear=2


Pb: spline=20, linear=22, overlap=20 (100%), only-spline=0, only-linear=2


Zn: spline=76, linear=79, overlap=76 (100%), only-spline=0, only-linear=3

=== pH form sensitivity summary ===
metal  n_spline  n_linear  overlap  pct_overlap
   As        79        95       79   100.000000
   Cd        30       118       30   100.000000
   Cr        66        68       65    98.484848
   Cu         8         8        8   100.000000
   Ni        41        43       41   100.000000
   Pb        20        22       20   100.000000
   Zn        76        79       76   100.000000


In [7]:
# === 4. Metal transformation sensitivity: log10 vs rank-normalization ===
# Rank-transform the metal concentration (robust to outliers/skew).
# If hit overlap is high, results are not sensitive to transformation choice.
from scipy.stats import rankdata as scipy_rankdata

cwm_vals = cwm_wide.values
rank_rows = []

for metal in METALS:
    if metal not in combined_idx.columns: continue
    raw = pd.to_numeric(combined_idx.reindex(base.index)[metal], errors='coerce').values
    
    # Log10 (standard)
    X_log = np.log10(np.where(raw > 0, raw, np.nan))
    Z1 = build_Z_spline(base, 1)
    valid = np.isfinite(X_log) & np.all(np.isfinite(Z1), axis=1)
    betas, se, t = fwl_all_kos(X_log[valid], cwm_vals[valid], Z1[valid])
    dof = max(valid.sum() - Z1.shape[1] - 1, 1)
    p = 2 * t_dist.sf(np.abs(t), df=dof)
    fp = np.isfinite(p); q = np.full(len(ko_ids), np.nan)
    if fp.sum() > 0:
        _, qv, _, _ = multipletests(p[fp], method='fdr_bh'); q[fp] = qv
    hits_log = set(np.array(ko_ids)[q < 0.05])
    
    # Rank-normalized
    raw_v = raw[valid]
    valid_raw = np.isfinite(raw_v) & (raw_v > 0)
    X_rank = np.full(valid.sum(), np.nan)
    r = scipy_rankdata(raw_v[valid_raw])
    n_r = valid_raw.sum()
    # inverse normal transform
    from scipy.stats import norm as scipy_norm
    X_rank[valid_raw] = scipy_norm.ppf((r - 0.5) / n_r)
    valid2 = np.isfinite(X_rank) & np.all(np.isfinite(Z1[valid]), axis=1)
    if valid2.sum() < 30: continue
    betas_r, se_r, t_r = fwl_all_kos(X_rank[valid2], cwm_vals[valid][valid2], Z1[valid][valid2])
    dof2 = max(valid2.sum() - Z1.shape[1] - 1, 1)
    p_r = 2 * t_dist.sf(np.abs(t_r), df=dof2)
    fp_r = np.isfinite(p_r); q_r = np.full(len(ko_ids), np.nan)
    if fp_r.sum() > 0:
        _, qvr, _, _ = multipletests(p_r[fp_r], method='fdr_bh'); q_r[fp_r] = qvr
    hits_rank = set(np.array(ko_ids)[q_r < 0.05])
    
    ov = len(hits_log & hits_rank)
    pct = 100 * ov / max(len(hits_log), 1)
    print(f'{metal}: log10={len(hits_log)}, rank={len(hits_rank)}, overlap={ov} ({pct:.0f}%)')
    rank_rows.append({'metal':metal,'n_log':len(hits_log),'n_rank':len(hits_rank),
                      'overlap':ov,'pct_overlap':pct})

rank_df = pd.DataFrame(rank_rows)
print('\n=== Metal transformation sensitivity ===')
print(rank_df.to_string(index=False))


As: log10=79, rank=13, overlap=13 (16%)


Cd: log10=30, rank=23, overlap=16 (53%)


Cr: log10=66, rank=0, overlap=0 (0%)


Cu: log10=8, rank=2, overlap=2 (25%)


Ni: log10=41, rank=0, overlap=0 (0%)


Pb: log10=20, rank=2, overlap=2 (10%)


Zn: log10=76, rank=9, overlap=9 (12%)

=== Metal transformation sensitivity ===
metal  n_log  n_rank  overlap  pct_overlap
   As     79      13       13    16.455696
   Cd     30      23       16    53.333333
   Cr     66       0        0     0.000000
   Cu      8       2        2    25.000000
   Ni     41       0        0     0.000000
   Pb     20       2        2    10.000000
   Zn     76       9        9    11.842105


In [8]:
# === 5. Collider sensitivity: L5-only hits that are NOT L1 hits ===
# Community composition at L5 (Shannon + phylum RA) is a potential collider:
# metal → community composition → KO expression.
# KOs that appear at L5 but NOT L1 may be false positives from collider induction.
# KOs that appear at BOTH L1 and L5 are the most robust.

for metal in METALS:
    l1_kos = set(nb02_L1[nb02_L1['metal']==metal]['ko_id'])
    l5_kos = set(nb02_L5[nb02_L5['metal']==metal]['ko_id'])
    l5_only = l5_kos - l1_kos    # potential collider artifacts
    l1_and_l5 = l1_kos & l5_kos  # robust hits (survive both controls)
    lost_at_l5 = l1_kos - l5_kos  # attenuated by community control (possibly mediation)
    print(f'{metal}: L1={len(l1_kos)}, L5={len(l5_kos)}, '
          f'L1∩L5={len(l1_and_l5)} ({100*len(l1_and_l5)/max(len(l1_kos),1):.0f}%), '
          f'L5-only={len(l5_only)}, L1-only={len(lost_at_l5)}')

print()
print('Interpretation:')
print('  L1∩L5: robust (appear before and after community control)')
print('  L1-only: attenuated at L5 — possibly mediated through community composition')
print('  L5-only: potential collider artifacts — metal affects community which affects KO')
print()
# Also check direction consistency between L1 and L5
for metal in ['As','Zn']:
    l1 = fdr_all[(fdr_all['level']=='L1')&(fdr_all['metal']==metal)].set_index('ko_id')
    l5 = fdr_all[(fdr_all['level']=='L5')&(fdr_all['metal']==metal)].set_index('ko_id')
    shared = l1.index.intersection(l5.index)
    if len(shared) > 0:
        concordant = (np.sign(l1.loc[shared,'beta']) == np.sign(l5.loc[shared,'beta'])).sum()
        print(f'{metal} L1∩L5 direction concordance: {concordant}/{len(shared)} ({100*concordant/len(shared):.0f}%)')


As: L1=41, L5=0, L1∩L5=0 (0%), L5-only=0, L1-only=41
Cd: L1=10, L5=0, L1∩L5=0 (0%), L5-only=0, L1-only=10
Cr: L1=38, L5=1, L1∩L5=1 (3%), L5-only=0, L1-only=37
Cu: L1=0, L5=0, L1∩L5=0 (0%), L5-only=0, L1-only=0
Ni: L1=38, L5=0, L1∩L5=0 (0%), L5-only=0, L1-only=38
Pb: L1=14, L5=2, L1∩L5=2 (14%), L5-only=0, L1-only=12
Zn: L1=76, L5=0, L1∩L5=0 (0%), L5-only=0, L1-only=76

Interpretation:
  L1∩L5: robust (appear before and after community control)
  L1-only: attenuated at L5 — possibly mediated through community composition
  L5-only: potential collider artifacts — metal affects community which affects KO

As L1∩L5 direction concordance: 4288/6557 (65%)
Zn L1∩L5 direction concordance: 4959/6557 (76%)


In [9]:
# Figures: summary of all sensitivity checks
from figure_style import grid_h

fig, axes = plt.subplots(2, 2, figsize=(FIGW['full'], ROW_H * 2))

# Panel A: pESS — effective n vs observed n
ax = axes[0, 0]
grid_h(ax)
x = np.arange(len(pess_df))
ax.bar(x - 0.2, pess_df['n'], 0.4, label='n observed', color=PALETTE[0], edgecolor='k', lw=0.5)
ax.bar(x + 0.2, pess_df['n_eff'], 0.4, label='n_eff (pESS)', color=PALETTE[1], edgecolor='k', lw=0.5)
ax.set_xticks(x); ax.set_xticklabels(pess_df['metal'])
ax.set_ylabel('Sample count'); ax.set_xlabel('Metal')
ax.set_title('pESS: spatial autocorrelation adjustment')
ax.legend(fontsize=8)
ax.annotate(f'Moran I range: [{pess_df["moran_I"].min():.2f}, {pess_df["moran_I"].max():.2f}]',
            xy=(0.02, 0.92), xycoords='axes fraction', fontsize=8)

# Panel B: Permutation test — observed vs null distribution (one metal)
ax = axes[0, 1]
grid_h(ax)
if len(perm_df) > 0:
    metals_sorted = perm_df.sort_values('n_obs', ascending=False)['metal'].tolist()
    colors = [PALETTE[0] if perm_df[perm_df['metal']==m]['perm_p'].values[0] < 0.002 else PALETTE[2]
              for m in metals_sorted]
    ax.bar(range(len(metals_sorted)),
           [perm_df[perm_df['metal']==m]['n_obs'].values[0] for m in metals_sorted],
           color=colors, edgecolor='k', lw=0.5, label='Observed FDR hits')
    ax.errorbar(range(len(metals_sorted)),
                [perm_df[perm_df['metal']==m]['null_mean'].values[0] for m in metals_sorted],
                yerr=[perm_df[perm_df['metal']==m]['null_sd'].values[0] for m in metals_sorted],
                fmt='D', color='gray', ms=4, lw=1.2, capsize=3, label='Null (500 perms)')
    ax.set_xticks(range(len(metals_sorted))); ax.set_xticklabels(metals_sorted)
    ax.set_ylabel('FDR hits (L1, q<0.05)'); ax.set_xlabel('Metal')
    ax.set_title('Permutation test (500 shuffles)')
    ax.legend(fontsize=8)

# Panel C: pH sensitivity — spline vs linear
ax = axes[1, 0]
grid_h(ax)
if len(ph_df) > 0:
    x2 = np.arange(len(ph_df))
    ax.bar(x2 - 0.2, ph_df['n_spline'], 0.4, label='Spline pH (standard)', color=PALETTE[0], edgecolor='k', lw=0.5)
    ax.bar(x2 + 0.2, ph_df['n_linear'], 0.4, label='Linear pH', color=PALETTE[3], edgecolor='k', lw=0.5)
    ax.set_xticks(x2); ax.set_xticklabels(ph_df['metal'])
    ax.set_ylabel('FDR hits (L1, q<0.05)'); ax.set_xlabel('Metal')
    ax.set_title('pH form sensitivity: spline vs linear')
    ax.legend(fontsize=8)

# Panel D: Metal transformation sensitivity
ax = axes[1, 1]
grid_h(ax)
if len(rank_df) > 0:
    x3 = np.arange(len(rank_df))
    ax.bar(x3 - 0.2, rank_df['n_log'], 0.4, label='log₁₀ (standard)', color=PALETTE[0], edgecolor='k', lw=0.5)
    ax.bar(x3 + 0.2, rank_df['n_rank'], 0.4, label='Rank-normal', color=PALETTE[4], edgecolor='k', lw=0.5)
    ax.set_xticks(x3); ax.set_xticklabels(rank_df['metal'])
    ax.set_ylabel('FDR hits (L1, q<0.05)'); ax.set_xlabel('Metal')
    ax.set_title('Metal transform: log₁₀ vs rank-normal')
    ax.legend(fontsize=8)

fig.suptitle('NB07: Sensitivity and robustness checks', y=1.02)
save(fig, FIGS / 'fig_nb07_sensitivity')


In [10]:
# === Summary ===
print('=== NB07 SENSITIVITY SUMMARY ===')
print()
print('1. pESS (spatial autocorrelation):')
for _, r in pess_df.iterrows():
    ratio = r['n_eff']/r['n']
    print(f'   {r["metal"]}: n={r["n"]}, n_eff={r["n_eff"]}, ratio={ratio:.2f}, Moran I={r["moran_I"]:.3f}')
print()
print('2. Permutation test (500x shuffle, L1):')
for _, r in perm_df.iterrows():
    sig = "***" if r['perm_p'] < 0.002 else ("*" if r['perm_p'] < 0.05 else "NS")
    print(f'   {r["metal"]}: obs={r["n_obs"]}, null={r["null_mean"]:.1f}±{r["null_sd"]:.1f}, p={r["perm_p"]:.4f} {sig}')
print()
print('3. pH form (spline vs linear):')
for _, r in ph_df.iterrows():
    print(f'   {r["metal"]}: {r["pct_overlap"]:.0f}% overlap ({r["overlap"]}/{r["n_spline"]})')
print()
print('4. Metal transform (log10 vs rank-normal):')
for _, r in rank_df.iterrows():
    print(f'   {r["metal"]}: {r["pct_overlap"]:.0f}% overlap ({r["overlap"]}/{r["n_log"]})')
print()
print('5. Collider check: see output above.')


=== NB07 SENSITIVITY SUMMARY ===

1. pESS (spatial autocorrelation):
   As: n=1143, n_eff=435, ratio=0.38, Moran I=0.448
   Cd: n=998, n_eff=430, ratio=0.43, Moran I=0.397
   Cr: n=1693, n_eff=593, ratio=0.35, Moran I=0.481
   Cu: n=1689, n_eff=764, ratio=0.45, Moran I=0.377
   Ni: n=1645, n_eff=722, ratio=0.44, Moran I=0.390
   Pb: n=1630, n_eff=778, ratio=0.48, Moran I=0.354
   Zn: n=1184, n_eff=513, ratio=0.43, Moran I=0.395

2. Permutation test (500x shuffle, L1):
   As: obs=79, null=9.0±47.1, p=0.0200 *
   Cd: obs=30, null=16.5±106.6, p=0.0640 NS
   Cr: obs=66, null=4.9±23.7, p=0.0220 *
   Cu: obs=8, null=3.7±48.5, p=0.0220 *
   Ni: obs=41, null=6.2±22.8, p=0.0440 *
   Pb: obs=20, null=8.7±28.9, p=0.0880 NS
   Zn: obs=76, null=9.7±54.3, p=0.0300 *

3. pH form (spline vs linear):
   As: 100% overlap (79/79)
   Cd: 100% overlap (30/30)
   Cr: 98% overlap (65/66)
   Cu: 100% overlap (8/8)
   Ni: 100% overlap (41/41)
   Pb: 100% overlap (20/20)
   Zn: 100% overlap (76/76)

4. Metal tr